# Notebook 04 — Aggregate docs_only Tables

**Part of:** Paper 3 reproducibility package.

**What this does:** Reads per-model intermediate CSVs and the audit table produced by
notebooks 01–03, then writes the five publication-ready `paper3_*_docs_only.csv` files.

**Inputs (all from `../results/` unless overridden):**
- `paper3_cross_model_audit_table.csv` — M1–M6 crossovers, hourglass, m6 rho/p
- `{model}_causal_swap_raw.csv` — raw swap data for bootstrap CIs
- `{model}_causal_swap_layer_summary.csv` — per-layer flip rates
- `{model}_m4_ablation_summary.csv` — for hourglass_no_l0 recompute (if available)
- `{model}_m6_per_prompt_convergence.csv` — for M6 entropy/convergence stats (if available)
- `../data/archived_inputs.csv` — frozen values for GPT-2/Gemma where source files unavailable
- `../config.yaml` — band definitions, bootstrap seed

**Outputs (written to `../results/`):**
1. `paper3_l0_diagnostics_docs_only.csv`
2. `paper3_invariance_summary_docs_only.csv`
3. `paper3_m6_stats_docs_only.csv`
4. `paper3_causal_effect_sizes_docs_only.csv`
5. `paper3_core_metrics_table.csv`

**No circular dependencies:** Missing per-model files are resolved from `archived_inputs.csv`,
which was populated from the original Colab run — not from the outputs of this notebook.
If a value cannot be found in either the source files or the archive, the notebook raises
a `RuntimeError` with a clear message rather than silently producing wrong values.

In [ ]:
import os, yaml, json
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.'
RESULTS_DIR   = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results'))
ARCHIVE_PATH  = os.path.join(NOTEBOOK_DIR, '..', 'data', 'archived_inputs.csv')
CONFIG_PATH   = os.path.join(NOTEBOOK_DIR, '..', 'config.yaml')

# Optional fallback directory for intermediate files not in results/.
# Set the environment variable PAPER3_DOWNLOADS_DIR if you have the raw per-model
# M4/M6 files stored elsewhere (e.g. a local Downloads folder).
# Leave unset to skip the fallback and rely on archived_inputs.csv instead.
DOWNLOADS_DIR = os.environ.get("PAPER3_DOWNLOADS_DIR", "")

# ── Config ────────────────────────────────────────────────────────────────────
cfg          = yaml.safe_load(open(CONFIG_PATH))
exp_cfg      = cfg['experiment']
LATE_SIZE    = exp_cfg['late_band_size']
EARLY_SIZE   = exp_cfg['early_band_size']
N_BOOT       = exp_cfg['n_bootstrap']
BOOT_SEED    = exp_cfg['bootstrap_seed']

# ── Model registry ────────────────────────────────────────────────────────────
# (file_key, audit_key, display_label, n_layers)
MODELS = [
    ('gpt2',       'gpt2',         'GPT-2 Small',  12),
    ('gemma2_2b',  'gemma2_2b',    'Gemma-2-2B',   26),
    ('qwen2_1_5b', 'qwen2.5-1.5b', 'Qwen2.5-1.5B', 28),
]

# ── Causal swap condition names ───────────────────────────────────────────────
COND_MAIN    = 'main_a_to_b'
COND_RANDOM  = 'ctrl_random_norm'
COND_SELF    = 'ctrl_self_b_to_b'
COND_SHUFFLE = 'ctrl_pos_shuffle'

# ── Archive (frozen values for GPT-2 & Gemma where source files are gone) ────
archive_df = pd.read_csv(ARCHIVE_PATH)

def get_archived(model_key, value_name):
    """Return float value from archived_inputs.csv, or raise RuntimeError."""
    row = archive_df[(archive_df['model_key'] == model_key) &
                     (archive_df['value_name'] == value_name)]
    if row.empty:
        raise RuntimeError(
            f"Value '{value_name}' for '{model_key}' not found in archive "
            f"AND source file is unavailable. Cannot proceed.\n"
            f"Solution: re-run notebook 01 for {model_key} and save the M4/M6 outputs."
        )
    return float(row.iloc[0]['value'])

def find_file(*candidates):
    """Return first existing path from candidates, or None."""
    return next((p for p in candidates if p and os.path.exists(p)), None)

print(f'Results dir:   {RESULTS_DIR}')
print(f'Downloads dir: {DOWNLOADS_DIR or "(not set — using archive only)"}')
print(f'Late band:     last {LATE_SIZE} layers')
print(f'Early band:    first {EARLY_SIZE} layers')
print(f'Bootstrap:     N={N_BOOT}, seed={BOOT_SEED}')


## Step 1 — Load Audit Table and Causal Swap Data

In [ ]:
# ── Audit table ───────────────────────────────────────────────────────────────
audit = pd.read_csv(
    os.path.join(RESULTS_DIR, 'paper3_cross_model_audit_table.csv')
).set_index('model')
print('Audit table loaded:', list(audit.index))

# ── Per-model causal swap files ───────────────────────────────────────────────
layer_sum = {}   # file_key -> DataFrame
raw_dfs   = {}   # file_key -> DataFrame

for file_key, audit_key, label, n_layers in MODELS:
    ls = find_file(
        os.path.join(RESULTS_DIR, f'{file_key}_causal_swap_layer_summary.csv'),
    )
    if ls:
        layer_sum[file_key] = pd.read_csv(ls)
        print(f'  [{label}] layer_summary: {len(layer_sum[file_key])} rows')
    else:
        print(f'  [{label}] WARNING: layer_summary missing — peak_layer will use audit table')

    raw = find_file(
        os.path.join(RESULTS_DIR, f'{file_key}_causal_swap_raw.csv'),
    )
    if raw:
        raw_dfs[file_key] = pd.read_csv(raw)
        print(f'  [{label}] raw_swap:      {len(raw_dfs[file_key])} rows')
    else:
        print(f'  [{label}] WARNING: raw_swap missing — bootstrap CIs cannot be computed')

# ── Per-model M4 ablation (for hourglass_no_l0 recompute) ────────────────────
m4_files = {}
for file_key, _, label, _ in MODELS:
    p = find_file(
        os.path.join(RESULTS_DIR,   f'{file_key}_m4_ablation_summary.csv'),
        os.path.join(DOWNLOADS_DIR, f'{file_key}_m4_ablation_summary.csv'),
        # Qwen uses hyphen naming in Downloads
        os.path.join(DOWNLOADS_DIR, 'qwen2.5-1.5b_m4_ablation_summary.csv')
        if file_key == 'qwen2_1_5b' else None,
    )
    m4_files[file_key] = p
    print(f'  [{label}] M4 ablation: {"FOUND" if p else "using archive"}')

# ── Per-model M6 per-prompt (for convergence stats) ───────────────────────────
m6_files = {}
for file_key, _, label, _ in MODELS:
    p = find_file(
        os.path.join(RESULTS_DIR,   f'{file_key}_m6_per_prompt_convergence.csv'),
        os.path.join(RESULTS_DIR,   'qwen2.5-1.5b_m6_per_prompt_convergence.csv')
        if file_key == 'qwen2_1_5b' else None,
        os.path.join(DOWNLOADS_DIR, f'{file_key}_m6_per_prompt.csv'),
    )
    m6_files[file_key] = p
    print(f'  [{label}] M6 per-prompt: {"FOUND" if p else "using archive"}')

## Step 2 — Bootstrap Utilities

In [ ]:
def get_band_layers(n_layers):
    return list(range(n_layers - LATE_SIZE, n_layers)), list(range(0, EARLY_SIZE))


def bootstrap_band_stats(raw_df, condition, late_layers, early_layers):
    """
    Pair-level percentile bootstrap for late/early band delta_margin.
    Algorithm:
      1. Filter to condition.
      2. For each of N_BOOT replicates, resample pair_idx with replacement.
      3. Compute mean delta_margin over band layers.
      4. Return observed mean + 2.5th/97.5th percentile CI.
    """
    rng  = np.random.default_rng(BOOT_SEED)
    df   = raw_df[raw_df['condition'] == condition].copy()
    pairs = df['pair_idx'].unique()
    n    = len(pairs)

    late_obs  = df[df['layer'].isin(late_layers)]['delta_margin'].mean()
    early_obs = df[df['layer'].isin(early_layers)]['delta_margin'].mean()

    boot_late = np.zeros(N_BOOT)
    boot_early = np.zeros(N_BOOT)
    for i in range(N_BOOT):
        idx    = rng.integers(0, n, size=n)
        sample = df[df['pair_idx'].isin(pairs[idx])]
        boot_late[i]  = sample[sample['layer'].isin(late_layers)]['delta_margin'].mean()
        boot_early[i] = sample[sample['layer'].isin(early_layers)]['delta_margin'].mean()

    return {
        'late_mean':   float(late_obs),
        'late_ci_lo':  float(np.percentile(boot_late,  2.5)),
        'late_ci_hi':  float(np.percentile(boot_late, 97.5)),
        'early_mean':  float(early_obs),
        'early_ci_lo': float(np.percentile(boot_early,  2.5)),
        'early_ci_hi': float(np.percentile(boot_early, 97.5)),
    }


def bootstrap_condition_diff(raw_df, cond_a, cond_b, late_layers):
    """Pair-level bootstrap for (cond_a − cond_b) in late band."""
    rng  = np.random.default_rng(BOOT_SEED)
    pa   = raw_df[(raw_df['condition'] == cond_a) & (raw_df['layer'].isin(late_layers))]\
               .groupby('pair_idx')['delta_margin'].mean()
    pb   = raw_df[(raw_df['condition'] == cond_b) & (raw_df['layer'].isin(late_layers))]\
               .groupby('pair_idx')['delta_margin'].mean()
    common = pa.index.intersection(pb.index)
    diffs  = (pa.loc[common] - pb.loc[common]).values
    n      = len(diffs)
    obs    = float(diffs.mean())
    boot   = np.array([diffs[rng.integers(0, n, n)].mean() for _ in range(N_BOOT)])
    return obs, float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))


print('Bootstrap utilities ready.')

## Step 3 — `paper3_l0_diagnostics_docs_only.csv`

Formula (from Cell 17 of `gpt2_rerun_confirmation_fast.ipynb`):
```
mid_band         = M4_ablation[ n//3 <= layer < 2*n//3 ]
mid_mean         = mean(mid_band.abl_kl_mean)
edge_no_l0       = M4_ablation at layer=1 UNION last-2 layers   (excludes L0)
edge_with_l0     = M4_ablation at layer=0,1 UNION last-2 layers
hourglass_no_l0  = mean(edge_no_l0.abl_kl_mean)   / (mid_mean + 1e-10)
hourglass_with_l0= mean(edge_with_l0.abl_kl_mean) / (mid_mean + 1e-10)
late_ramp_ratio  = mean(M5.update_norm_mean in late_layers) / mean(M5.update_norm_mean in mid_band)
```
`hourglass_with_l0` = `hourglass_edge_mid_ratio` in the audit table (used as cross-check).

In [ ]:
rows_l0 = []

for file_key, audit_key, label, n_layers in MODELS:
    row_a      = audit.loc[audit_key]
    with_l0    = float(row_a['hourglass_edge_mid_ratio'])  # from audit table
    late_ramp  = float(row_a['late_ramp_ratio'])            # from audit table

    m4_path = m4_files[file_key]
    if m4_path:
        abl = pd.read_csv(m4_path).sort_values('layer').reset_index(drop=True)
        mid = abl[(abl['layer'] >= n_layers // 3) & (abl['layer'] < 2 * n_layers // 3)]
        mid_mean   = mid['abl_kl_mean'].mean()
        edge_no_l0 = pd.concat([abl.iloc[[1]], abl.tail(2)])
        no_l0  = edge_no_l0['abl_kl_mean'].mean() / (mid_mean + 1e-10)
        source = 'recomputed'
    else:
        no_l0  = get_archived(file_key, 'hourglass_ratio_no_l0')
        source = '[archived]'

    multiplier = with_l0 / no_l0
    rows_l0.append({
        'model_key':               file_key,
        'model':                   label,
        'hourglass_ratio_no_l0':   no_l0,
        'hourglass_ratio_with_l0': with_l0,
        'l0_inflation_multiplier': multiplier,
        'late_ramp_ratio':         late_ramp,
    })
    print(f'{label:14s} | no_l0={no_l0:.4f} | with_l0={with_l0:.4f} '
          f'| mult={multiplier:.3f} | late_ramp={late_ramp:.4f} | {source}')

df_l0 = pd.DataFrame(rows_l0)
df_l0.to_csv(os.path.join(RESULTS_DIR, 'paper3_l0_diagnostics_docs_only.csv'), index=False)
print('Saved: paper3_l0_diagnostics_docs_only.csv')

## Step 4 — `paper3_invariance_summary_docs_only.csv`

Formula:
```
mean_boundary_pct_depth = mean(m1, m2, m3, m4, m5 layers) / (n_layers − 1) × 100
causal_peak_pct_depth   = argmax(flip_to_a_rate for main_a_to_b) / (n_layers − 1) × 100
```

In [ ]:
rows_inv = []

for (file_key, audit_key, label, n_layers), l0_row in zip(MODELS, rows_l0):
    row_a = audit.loc[audit_key]

    boundary_layers = [float(row_a[f'm{i}_layer']) for i in range(1, 6)]
    mean_boundary_pct = (np.mean(boundary_layers) / (n_layers - 1)) * 100

    if file_key in layer_sum:
        ls = layer_sum[file_key]
        main_ls = ls[ls['condition'] == COND_MAIN]
        peak_layer = int(main_ls.loc[main_ls['flip_to_a_rate'].idxmax(), 'layer'])
    else:
        peak_layer = n_layers - 1  # safe default
        print(f'  [{label}] WARNING: using last layer as causal peak (no layer_summary)')

    causal_peak_pct = (peak_layer / (n_layers - 1)) * 100

    rows_inv.append({
        'model':                   label,
        'n_layers':                n_layers,
        'mean_boundary_pct_depth': mean_boundary_pct,
        'causal_peak_pct_depth':   causal_peak_pct,
        'hourglass_no_l0':         l0_row['hourglass_ratio_no_l0'],
        'late_ramp_ratio':         float(row_a['late_ramp_ratio']),
        'm6_rho':                  float(row_a['m6_spearman_rho']),
        'm6_p':                    float(row_a['m6_spearman_p']),
    })
    print(f'{label:14s} | boundary%={mean_boundary_pct:.2f} | peak%={causal_peak_pct:.2f} (L{peak_layer})')

df_inv = pd.DataFrame(rows_inv)
df_inv.to_csv(os.path.join(RESULTS_DIR, 'paper3_invariance_summary_docs_only.csv'), index=False)
print('Saved: paper3_invariance_summary_docs_only.csv')

## Step 5 — `paper3_m6_stats_docs_only.csv`

In [ ]:
rows_m6 = []

for file_key, audit_key, label, n_layers in MODELS:
    row_a = audit.loc[audit_key]
    m6_path = m6_files[file_key]

    if m6_path:
        df_m6 = pd.read_csv(m6_path)
        n_prompts = len(df_m6)
        rho, pval  = stats.spearmanr(df_m6['final_entropy'], df_m6['convergence_layer'])
        conv_mean  = float(df_m6['convergence_layer'].mean())
        conv_std   = float(df_m6['convergence_layer'].std(ddof=1))
        ent_mean   = float(df_m6['final_entropy'].mean())
        ent_std    = float(df_m6['final_entropy'].std(ddof=1))
        source = f'computed from {os.path.basename(m6_path)}'
    else:
        n_prompts = 15
        rho       = get_archived(file_key, 'm6_spearman_rho')
        pval      = get_archived(file_key, 'm6_spearman_p')
        conv_mean = get_archived(file_key, 'convergence_layer_mean')
        conv_std  = get_archived(file_key, 'convergence_layer_std')
        ent_mean  = get_archived(file_key, 'entropy_mean')
        ent_std   = get_archived(file_key, 'entropy_std')
        source = '[archived]'

    rows_m6.append({
        'model_key':                           file_key,
        'model':                               label,
        'n_prompts':                           n_prompts,
        'spearman_rho_entropy_vs_convergence': rho,
        'p_value':                             pval,
        'convergence_layer_mean':              conv_mean,
        'convergence_layer_std':               conv_std,
        'entropy_mean':                        ent_mean,
        'entropy_std':                         ent_std,
    })
    print(f'{label:14s} | rho={rho:.4f} p={pval:.4f} '
          f'| conv={conv_mean:.2f}±{conv_std:.2f} | ent={ent_mean:.3f} | {source}')

df_m6_out = pd.DataFrame(rows_m6)
df_m6_out.to_csv(os.path.join(RESULTS_DIR, 'paper3_m6_stats_docs_only.csv'), index=False)
print('Saved: paper3_m6_stats_docs_only.csv')

## Step 6 — `paper3_causal_effect_sizes_docs_only.csv`

Bootstrap procedure:
1. Late band = last `late_band_size` layers; early band = first `early_band_size` layers.
2. Pair-level resampling with replacement, `n_bootstrap` replicates, seed from config.
3. `overall = PASS` if `late_delta_ci_lo > 0`.

In [ ]:
rows_ce = []

for file_key, audit_key, label, n_layers in MODELS:
    if file_key not in raw_dfs:
        raise RuntimeError(
            f'raw_swap file missing for {label}. '
            f'Run notebook 02 for this model first, or place the file at:\n'
            f'  {os.path.join(RESULTS_DIR, file_key + "_causal_swap_raw.csv")}'
        )

    print(f'\n{label} ({n_layers} layers)...')
    late_layers, early_layers = get_band_layers(n_layers)
    raw = raw_dfs[file_key]
    ls  = layer_sum.get(file_key)

    # Peak layer
    if ls is not None:
        main_ls    = ls[ls['condition'] == COND_MAIN]
        peak_layer = int(main_ls.loc[main_ls['flip_to_a_rate'].idxmax(), 'layer'])
    else:
        peak_layer = n_layers - 1

    # Bootstrap: main condition late/early bands
    print('  Bootstrapping main condition...', end=' ', flush=True)
    bs = bootstrap_band_stats(raw, COND_MAIN, late_layers, early_layers)
    print('done.')

    # Bootstrap: condition differences (late band)
    print('  Bootstrapping condition diffs...', end=' ', flush=True)
    diff_rand = bootstrap_condition_diff(raw, COND_MAIN, COND_RANDOM,  late_layers)
    diff_self = bootstrap_condition_diff(raw, COND_MAIN, COND_SELF,    late_layers)
    diff_shuf = bootstrap_condition_diff(raw, COND_MAIN, COND_SHUFFLE, late_layers)
    print('done.')

    overall = 'PASS' if bs['late_ci_lo'] > 0 else 'FAIL'
    print(f'  late_delta={bs["late_mean"]:.4f} '
          f'CI=[{bs["late_ci_lo"]:.4f}, {bs["late_ci_hi"]:.4f}] -> {overall}')

    rows_ce.append({
        'model_key':                    file_key,
        'model':                        label,
        'overall':                      overall,
        'peak_layer':                   peak_layer,
        'expected_late_lo':             late_layers[0],
        'expected_late_hi':             late_layers[-1],
        'late_delta_mean':              bs['late_mean'],
        'late_delta_ci_lo':             bs['late_ci_lo'],
        'late_delta_ci_hi':             bs['late_ci_hi'],
        'early_delta_mean':             bs['early_mean'],
        'early_delta_ci_lo':            bs['early_ci_lo'],
        'early_delta_ci_hi':            bs['early_ci_hi'],
        'main_minus_random_mean':       diff_rand[0],
        'main_minus_random_ci_lo':      diff_rand[1],
        'main_minus_random_ci_hi':      diff_rand[2],
        'main_minus_self_mean':         diff_self[0],
        'main_minus_self_ci_lo':        diff_self[1],
        'main_minus_self_ci_hi':        diff_self[2],
        'main_minus_pos_shuffle_mean':  diff_shuf[0],
        'main_minus_pos_shuffle_ci_lo': diff_shuf[1],
        'main_minus_pos_shuffle_ci_hi': diff_shuf[2],
    })

df_ce = pd.DataFrame(rows_ce)
df_ce.to_csv(os.path.join(RESULTS_DIR, 'paper3_causal_effect_sizes_docs_only.csv'), index=False)
print('\nSaved: paper3_causal_effect_sizes_docs_only.csv')

## Step 7 — `paper3_core_metrics_table.csv`

In [ ]:
ce_by_key = {r['model_key']: r for r in rows_ce}
rows_cm = []

for (file_key, audit_key, label, n_layers), l0_row in zip(MODELS, rows_l0):
    row_a = audit.loc[audit_key]
    ce    = ce_by_key.get(file_key, {})

    rows_cm.append({
        'model':                         file_key,
        'n_layers':                      n_layers,
        'raw_early_top1':                float(row_a['raw_early_top1']),
        'raw_late_top1':                 float(row_a['raw_late_top1']),
        'cast_early_top1':               float(row_a['cast_early_top1']),
        'raw_l50_layer':                 int(row_a['raw_top1_50_layer']),
        'raw_l90_layer':                 int(row_a['raw_top1_90_layer']),
        'm3_crossover_0p5':              int(row_a['m3_layer']),
        'hourglass_no_l0':               l0_row['hourglass_ratio_no_l0'],
        'late_ramp_ratio':               float(row_a['late_ramp_ratio']),
        'causal_overall':                ce.get('overall', 'not_run'),
        'causal_peak_layer':             ce.get('peak_layer', None),
        'causal_late_mean':              ce.get('late_delta_mean', None),
        'causal_early_mean':             ce.get('early_delta_mean', None),
        'causal_main_minus_random_mean': ce.get('main_minus_random_mean', None),
    })

df_cm = pd.DataFrame(rows_cm)
df_cm.to_csv(os.path.join(RESULTS_DIR, 'paper3_core_metrics_table.csv'), index=False)
print('Saved: paper3_core_metrics_table.csv')
print(df_cm.to_string(index=False))

## Step 8 — Spot-check Verification

In [ ]:
import math

# ── 1. Magnitude checks ───────────────────────────────────────────────────────
# Continuous values: ±0.5% relative tolerance.
# Integer layer values: ±1 layer absolute tolerance.
MAGNITUDE_CHECKS = [
    # (file, id_col, id_val, check_col, expected, is_layer)
    ("paper3_l0_diagnostics_docs_only.csv",    "model_key", "gpt2",        "hourglass_ratio_no_l0",                    3.1987,   False),
    ("paper3_l0_diagnostics_docs_only.csv",    "model_key", "gemma2_2b",   "hourglass_ratio_with_l0",                 76.532,   False),
    ("paper3_l0_diagnostics_docs_only.csv",    "model_key", "qwen2_1_5b",  "late_ramp_ratio",                          4.264,   False),
    ("paper3_l0_diagnostics_docs_only.csv",    "model_key", "gpt2",        "l0_inflation_multiplier",                  3.459,   False),
    ("paper3_invariance_summary_docs_only.csv","model",     "GPT-2 Small", "mean_boundary_pct_depth",                  72.73,   False),
    ("paper3_invariance_summary_docs_only.csv","model",     "Gemma-2-2B",  "mean_boundary_pct_depth",                  50.40,   False),
    ("paper3_invariance_summary_docs_only.csv","model",     "GPT-2 Small", "causal_peak_pct_depth",                   100.0,   False),
    ("paper3_invariance_summary_docs_only.csv","model",     "Qwen2.5-1.5B","causal_peak_pct_depth",                    92.59,  False),
    ("paper3_invariance_summary_docs_only.csv","model",     "Qwen2.5-1.5B","m6_rho",                                   0.3795,  False),
    ("paper3_m6_stats_docs_only.csv",          "model_key", "gpt2",        "spearman_rho_entropy_vs_convergence",      0.7314,  False),
    ("paper3_m6_stats_docs_only.csv",          "model_key", "gpt2",        "p_value",                                  0.00194, False),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gpt2",        "late_delta_mean",                          2.5137,  False),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gemma2_2b",   "late_delta_mean",                         11.687,   False),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","qwen2_1_5b",  "late_delta_mean",                         10.379,   False),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gpt2",        "peak_layer",                              11,       True),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gemma2_2b",   "peak_layer",                              25,       True),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","qwen2_1_5b",  "peak_layer",                              25,       True),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gpt2",        "expected_late_lo",                         8,       True),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","gemma2_2b",   "expected_late_lo",                        22,       True),
    ("paper3_causal_effect_sizes_docs_only.csv","model_key","qwen2_1_5b",  "expected_late_lo",                        24,       True),
]

print("=== 1. Magnitude checks ===")
all_pass = True
for fname, id_col, id_val, col, expected, is_layer in MAGNITUDE_CHECKS:
    df     = pd.read_csv(os.path.join(RESULTS_DIR, fname))
    row    = df[df[id_col] == id_val]
    if row.empty:
        print(f"  MISSING   {id_val:12s}  {col}")
        all_pass = False
        continue
    actual = float(row.iloc[0][col])
    if is_layer:
        ok = abs(actual - expected) <= 1
    else:
        ok = math.isclose(actual, expected, rel_tol=0.005)
    status = "PASS" if ok else "MISMATCH"
    if not ok:
        all_pass = False
    tol_str = "±1 layer" if is_layer else "±0.5%"
    print(f"  {status:8s}  {id_val:12s}  {col}: got={actual:.4f}  expected={expected}  ({tol_str})")

# ── 2. Behavioral parity checks ───────────────────────────────────────────────
print("\n=== 2. Behavioral parity — ordering ===")

ce = pd.read_csv(os.path.join(RESULTS_DIR, "paper3_causal_effect_sizes_docs_only.csv")).set_index("model_key")
l0 = pd.read_csv(os.path.join(RESULTS_DIR, "paper3_l0_diagnostics_docs_only.csv")).set_index("model_key")

ordering_checks = [
    # (description, value_a, label_a, value_b, label_b, expected: a > b)
    ("late_delta Gemma > GPT-2",
     ce.loc["gemma2_2b", "late_delta_mean"], "Gemma",
     ce.loc["gpt2",      "late_delta_mean"], "GPT-2", True),
    ("late_delta Qwen > GPT-2",
     ce.loc["qwen2_1_5b","late_delta_mean"], "Qwen",
     ce.loc["gpt2",      "late_delta_mean"], "GPT-2", True),
    ("hourglass_with_l0 Gemma > GPT-2",
     l0.loc["gemma2_2b", "hourglass_ratio_with_l0"], "Gemma",
     l0.loc["gpt2",      "hourglass_ratio_with_l0"], "GPT-2", True),
]
for desc, va, la, vb, lb, expect_gt in ordering_checks:
    ok = (va > vb) == expect_gt
    if not ok: all_pass = False
    print(f"  {'PASS' if ok else 'FAIL':8s}  {desc}: {la}={va:.3f}  {lb}={vb:.3f}")

print("\n=== 3. Behavioral parity — signs ===")
for file_key, _, label, _ in MODELS:
    if file_key not in ce.index: continue
    row = ce.loc[file_key]
    checks = [
        ("late_delta_ci_lo > 0 (PASS criterion)",       row["late_delta_ci_lo"]          > 0),
        ("late_delta_mean > early_delta_mean",           row["late_delta_mean"]           > row["early_delta_mean"]),
        ("main_minus_random_mean > 0",                   row["main_minus_random_mean"]    > 0),
        ("main_minus_self_mean > 0",                     row["main_minus_self_mean"]      > 0),
        ("main_minus_pos_shuffle_mean > 0",              row["main_minus_pos_shuffle_mean"]> 0),
    ]
    for desc, ok in checks:
        if not ok: all_pass = False
        print(f"  {'PASS' if ok else 'FAIL':8s}  {label}: {desc}")

print("\n=== 4. Behavioral parity — trends ===")
inv = pd.read_csv(os.path.join(RESULTS_DIR, "paper3_invariance_summary_docs_only.csv")).set_index("model")
trend_checks = [
    ("GPT-2: causal peak depth > mean boundary depth",
     float(inv.loc["GPT-2 Small","causal_peak_pct_depth"]) > float(inv.loc["GPT-2 Small","mean_boundary_pct_depth"])),
    ("Gemma: causal peak depth > mean boundary depth",
     float(inv.loc["Gemma-2-2B","causal_peak_pct_depth"])  > float(inv.loc["Gemma-2-2B","mean_boundary_pct_depth"])),
    ("Qwen: causal peak depth > mean boundary depth",
     float(inv.loc["Qwen2.5-1.5B","causal_peak_pct_depth"])> float(inv.loc["Qwen2.5-1.5B","mean_boundary_pct_depth"])),
    ("GPT-2: l0_inflation_multiplier > 1",
     float(l0.loc["gpt2","l0_inflation_multiplier"]) > 1),
    ("Gemma: l0_inflation_multiplier > GPT-2",
     float(l0.loc["gemma2_2b","l0_inflation_multiplier"]) > float(l0.loc["gpt2","l0_inflation_multiplier"])),
]
for desc, ok in trend_checks:
    if not ok: all_pass = False
    print(f"  {'PASS' if ok else 'FAIL':8s}  {desc}")

print()
if all_pass:
    print("ALL CHECKS PASSED — outputs match original run within acceptance criteria.")
else:
    print("WARNING: One or more checks FAILED. Review MISMATCH/FAIL lines above.")
